In [ ]:
from collections import Counter
from pathlib import Path
import sys

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "src"))

from data.datasets import get_dataset
from data.splits import get_train_val_split

dataset_name = "Oxford-IIIT Pet classification"
data_dir = repo_root / "data" / "raw" / "iit_pet"

# Keep samples raw until statistics have been computed on the training split.
trainval_dataset = get_dataset(
    split="trainval", dataset=dataset_name, data_dir=data_dir, download=False
)
raw_test_dataset = get_dataset(
    split="test", dataset=dataset_name, data_dir=data_dir, download=False
)

raw_train_dataset, raw_val_dataset = get_train_val_split(
    trainval_dataset, ratio=0.2, seed=42, labels=trainval_dataset._labels
)

assert set(raw_train_dataset.indices).isdisjoint(raw_val_dataset.indices)
for name, subset in (("Train", raw_train_dataset), ("Validation", raw_val_dataset)):
    counts = Counter(trainval_dataset._labels[i] for i in subset.indices)
    print(
        f"{name}: {len(subset)} images, {len(counts)} classes "
        f"({min(counts.values())}-{max(counts.values())} per class)"
    )
print(f"Test: {len(raw_test_dataset)} images (official held-out set)")
image, label = raw_train_dataset[0]
print(f"Example: image size={image.size}, label={label}")


In [ ]:
from functools import partial
from torch.utils.data import Subset
from data.datasets import calculate_mean_std
from data.dataloaders import get_dataloader
from data.transforms import image_transform

NUM_WORKERS = 4
BATCH_SIZE = 32
IMAGE_SIZE = (224, 224)

# Estimate RGB statistics using raw training images only.
mean, std = calculate_mean_std(raw_train_dataset, size=IMAGE_SIZE)
print("Train-only RGB mean/std:", mean, std)

train_transform = partial(image_transform, mean=mean, std=std, train=True, size=IMAGE_SIZE)
eval_transform = partial(image_transform, mean=mean, std=std, train=False, size=IMAGE_SIZE)

train_dataset = Subset(
    get_dataset("trainval", dataset_name, data_dir, transform=train_transform),
    raw_train_dataset.indices,
)
val_dataset = Subset(
    get_dataset("trainval", dataset_name, data_dir, transform=eval_transform),
    raw_val_dataset.indices,
)
test_dataset = get_dataset("test", dataset_name, data_dir, transform=eval_transform)

train_dataloader = get_dataloader(
    train_dataset, split="train", batch_size=BATCH_SIZE, num_workers=NUM_WORKERS
)
val_dataloader = get_dataloader(
    val_dataset, split="val", batch_size=BATCH_SIZE, num_workers=NUM_WORKERS
)
test_dataloader = get_dataloader(
    test_dataset, split="test", batch_size=BATCH_SIZE, num_workers=NUM_WORKERS
)

for split_name, loader in (
    ("train", train_dataloader), ("validation", val_dataloader), ("test", test_dataloader)
):
    images, labels = next(iter(loader))
    print(f"{split_name}: images={tuple(images.shape)}, labels={tuple(labels.shape)}")


In [3]:
from models import get_classification_model

convnext_model = get_classification_model("convnext_tiny", pretrained=True, num_classes=37)
swin_transformer_model = get_classification_model("swin_tiny_patch4_window7_224", pretrained=True, num_classes=37)
resnet50_model = get_classification_model("resnet50", pretrained=True, num_classes=37)

/Users/kasrasadatsharifi/Library/Caches/pypoetry/virtualenvs/convnexts-2-MddwQV-py3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Head-only classifier training (freeze the pretrained backbone)


In [ ]:
import torch
from evaluation import train_one_epoch, validate
from utils.io import save_checkpoint, save_json
from visualization import plot_training_history

NUM_EPOCHS = 10
DEVICE = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
CRITERION = torch.nn.CrossEntropyLoss()
LEARNING_RATE = 1e-4

# Head-only training: freeze all pretrained features explicitly.
# This creates a NEW experiment; previous whole-network runs are not comparable.
models = {
    "ConvNeXt Tiny": ("convnext_tiny", convnext_model),
    "Swin Transformer Tiny": ("swin_tiny", swin_transformer_model),
    "ResNet-50": ("resnet50", resnet50_model),
}

training_runs = {}
for model_name, (model_id, model) in models.items():
    print(f"Training classification head: {model_name}")
    model.to(DEVICE)

    for parameter in model.parameters():
        parameter.requires_grad = False
    for parameter in model.get_classifier().parameters():
        parameter.requires_grad = True

    optimizer = torch.optim.AdamW(
        (p for p in model.parameters() if p.requires_grad), lr=LEARNING_RATE
    )
    result_dir = repo_root / "results" / "classification" / model_id
    history = []
    best_val_accuracy = -1.0
    best_val_loss = float("inf")
    best_epoch = None

    for epoch in range(1, NUM_EPOCHS + 1):
        train_loss, train_acc = train_one_epoch(
            model, train_dataloader, CRITERION, optimizer, DEVICE, head_only=True
        )
        val_loss, val_acc = validate(model, val_dataloader, CRITERION, DEVICE)
        history.append({
            "epoch": epoch, "train_loss": train_loss, "train_accuracy": train_acc,
            "val_loss": val_loss, "val_accuracy": val_acc,
        })
        print(
            f"Epoch {epoch}/{NUM_EPOCHS}: train_loss={train_loss:.4f}, "
            f"train_acc={train_acc:.4f}, val_loss={val_loss:.4f}, val_acc={val_acc:.4f}"
        )

        # Prefer higher validation accuracy, then lower loss if tied.
        if val_acc > best_val_accuracy or (
            val_acc == best_val_accuracy and val_loss < best_val_loss
        ):
            best_epoch = epoch
            best_val_accuracy = val_acc
            best_val_loss = val_loss
            save_checkpoint(
                result_dir / "best.pt", model, optimizer, epoch,
                {"loss": val_loss, "accuracy": val_acc},
                metadata={
                    "model_name": model_name, "model_id": model_id,
                    "dataset": dataset_name, "training_strategy": "head_only",
                    "learning_rate": LEARNING_RATE, "image_size": IMAGE_SIZE,
                    "normalization_mean": mean, "normalization_std": std,
                    "split_seed": 42,
                },
            )

        save_json(history, result_dir / "training_history.json")

    training_runs[model_name] = {
        "model_id": model_id, "history": history,
        "best_epoch": best_epoch, "best_val_accuracy": best_val_accuracy,
        "best_val_loss": best_val_loss,
    }
    plot_training_history(history, model_name)
    print(f"Best {model_name}: epoch={best_epoch}, val_acc={best_val_accuracy:.4f}")


In [ ]:
from evaluation import evaluate
from utils.io import load_checkpoint, save_json, save_predictions
from visualization import display_metrics, plot_confusion_matrix

# Evaluate only the BEST validation checkpoint of each model on held-out test data.
test_results = {}
test_predictions = {}
for model_name, (model_id, model) in models.items():
    result_dir = repo_root / "results" / "classification" / model_id
    checkpoint = load_checkpoint(result_dir / "best.pt", model, device=DEVICE)
    metrics, predictions = evaluate(
        model, test_dataloader, CRITERION, DEVICE, return_predictions=True
    )
    test_results[model_name] = metrics
    test_predictions[model_name] = predictions
    save_json(
        {
            "model": model_name,
            "checkpoint_epoch": checkpoint["epoch"],
            "best_validation": checkpoint["validation_metrics"],
            "test_metrics": metrics,
        },
        result_dir / "test_metrics.json",
    )
    save_predictions(predictions, result_dir / "test_predictions.npz")

comparison = display_metrics(test_results)
save_json(test_results, repo_root / "results" / "classification" / "comparison.json")

# Run these selectively when inspecting classification errors:
# for name, result in test_results.items():
#     plot_confusion_matrix(result["confusion_matrix"], name, class_names=test_dataset.classes)


# fine tune more layers